<a href="https://colab.research.google.com/github/mahashabdekedar/Quantum_Computing_Projects/blob/main/MultilayerQNNStripesClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup
!pip install numpy
!pip install scipy
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit_ibm_runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/54

We take black and white 3x3 images representing stripes (horizontal) or bars (vertical), and attempt to classify them using a quantum neural network. We choose our network to be a set of rotations, followed by a layer of CZ gates, followed by another group of rotations, and then measurement. We flatten images into arrays of length 9, where array[0] represents the top left qubit and array[2] represents the top right. Let $\vert 1\rangle$ be the correct label for a stripe, and $\vert 0\rangle$ be the correct label for a bar.

In [ ]:
# Generate a training set
import numpy as np

# Stripes
stripes = []

for i in range(3):
  blank = np.ones((3, 3), np.int8)
  blank[i,:] = -1
  stripes.append(blank.flatten())
stripes.extend([mat * -1 for mat in stripes])

# Bars
bars = []

for i in range(3):
  blank = np.ones((3, 3), np.int8)
  blank[:,i] = -1
  bars.append(blank.flatten())

bars.extend([mat * -1 for mat in bars])

features = bars + stripes

# labels
labels  = np.concatenate((np.ones(len(bars)), -np.ones(len(stripes))))


We'll add some noise to our dataset, and holdout some examples for validation

In [ ]:
import random

rng = random.Random(42)
np_rng = np.random.default_rng()

# Let's add some noise to our data

noisy_features = []
noisy_labels = []
for feat, lab in zip(features, labels):
  for i in range(3):
    noisy_features.append(feat + np_rng.normal(loc=0, scale = 0.1, size=9))
    noisy_labels.append(lab)

# Holdout roughly 20%
holdout = set(rng.sample(range(0, len(noisy_features)), 8))
use = set(range(len(noisy_features))).difference(holdout)


x_train = [noisy_features[x] for x in use]
x_validate = [noisy_features[x] for x in holdout]

y_train = [noisy_labels[x] for x in use]
y_validate = [noisy_labels[x] for x in holdout]



# rng = random.Random(42)
# while len(holdout) != 2:
#   holdout.add(rng.randint(0, 11))

# use = set(range(12)).difference(holdout)

# x_train = [features[x] for x in use]
# y_train = [labels[x] for x in use]

# x_validate = [features[x] for x in holdout]
# y_validate = [labels[x] for x in holdout]

# print(x_validate)
# print(y_validate)



Now for the actual circuit. We'll setup and transpile a parameterized version, and assign values to the parameters later. Note that we choose the order of rotations y, z, x because if z were first or last it would have no effect either on the input basis state or on the measurement outcome of the output state. The order of y and x is arbitrary. Furthermore we choose CZ over CX because they commute.

We'll attach an encoder to pass the correct input state. We choose to encode input states using an x rotation on every qubit, with magnitude corresponding the value of the matching pixel. So if pixel 2 takes value 1, we rotate by 1 radian. Note that we could scale by pi to make a value of 1 correspond to input state $|1\rangle$, but then -1.01 and 0.99 would look identical, so we refrain.

In [ ]:
from qiskit.circuit import ParameterVector, QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import time



def set_params(parameters, qc, param_vec_theta):
  return qc.assign_parameters(dict(zip(param_vec_theta, parameters)))

def set_encoding(arr, qc, param_vec_phi):
  return qc.assign_parameters(dict(zip(param_vec_phi, arr)))

# Build the circuit
def buildCircuit(num_layers=1):
  qc = QuantumCircuit(9, 1)
  phi = ParameterVector("phi", 9)
  theta = ParameterVector("theta", 27*num_layers + 3)

  # Encoding Layer
  for i in range(9):
    qc.rx(phi[i], i)

  for i in range(num_layers):
    # Add a layer of rotations
    for j in range(9):
      qc.ry(theta[27*i + 3*j], j)
      qc.rz(theta[27*i + 3*j + 1], j)
      qc.rx(theta[27*i + 3*j + 2], j)

    # Add an entanglement layer
    qc.cz(0, 1)
    qc.cz(1, 2)
    qc.cz(3, 4)
    qc.cz(4, 5)
    qc.cz(6, 7)
    qc.cz(7, 8)
    qc.cz(0, 3)
    qc.cz(1, 4)
    qc.cz(2, 5)
    qc.cz(3, 6)
    qc.cz(4, 7)
    qc.cz(5, 8)

  # Add last layer of rotations
  qc.ry(theta[27*num_layers], 4)
  qc.rz(theta[27*num_layers + 1], 4)
  qc.rx(theta[27*num_layers + 2], 4)

  # Measurement
  qc.measure(qubit=4, cbit=0)

  return qc, phi, theta
main, phi_main, theta_main = buildCircuit(3)
main.draw()

┌────────────┐ ┌──────────────┐ ┌──────────────┐ ┌──────────────┐         »
q_0: ┤ Rx(phi[0]) ├─┤ Ry(theta[0]) ├─┤ Rz(theta[1]) ├─┤ Rx(theta[2]) ├─■─────■─»
     ├────────────┤ ├──────────────┤ ├──────────────┤ ├──────────────┤ │     │ »
q_1: ┤ Rx(phi[1]) ├─┤ Ry(theta[3]) ├─┤ Rz(theta[4]) ├─┤ Rx(theta[5]) ├─■──■──┼─»
     ├────────────┤ ├──────────────┤ ├──────────────┤ ├──────────────┤    │  │ »
q_2: ┤ Rx(phi[2]) ├─┤ Ry(theta[6]) ├─┤ Rz(theta[7]) ├─┤ Rx(theta[8]) ├────■──┼─»
     ├────────────┤ ├──────────────┤┌┴──────────────┤┌┴──────────────┤       │ »
q_3: ┤ Rx(phi[3]) ├─┤ Ry(theta[9]) ├┤ Rz(theta[10]) ├┤ Rx(theta[11]) ├─■─────■─»
     ├────────────┤┌┴──────────────┤├───────────────┤├───────────────┤ │       »
q_4: ┤ Rx(phi[4]) ├┤ Ry(theta[12]) ├┤ Rz(theta[13]) ├┤ Rx(theta[14]) ├─■──■────»
     ├────────────┤├───────────────┤├───────────────┤├───────────────┤    │    »
q_5: ┤ Rx(phi[5]) ├┤ Ry(theta[15]) ├┤ Rz(theta[16]) ├┤ Rx(theta[17]) ├────■────»
     ├────────────┤├───────────────┤├───────────────┤├───────────────┤         »
q_6: ┤ Rx(phi[6]) ├┤ Ry(theta[18]) ├┤ Rz(theta[19]) ├┤ Rx(theta[20]) ├─■───────»
     ├────────────┤├───────────────┤├───────────────┤├───────────────┤ │       »
q_7: ┤ Rx(phi[7]) ├┤ Ry(theta[21]) ├┤ Rz(theta[22]) ├┤ Rx(theta[23]) ├─■──■────»
     ├────────────┤├───────────────┤├───────────────┤├───────────────┤    │    »
q_8: ┤ Rx(phi[8]) ├┤ Ry(theta[24]) ├┤ Rz(theta[25]) ├┤ Rx(theta[26]) ├────■────»
     └────────────┘└───────────────┘└───────────────┘└───────────────┘         »
c: 1/══════════════════════════════════════════════════════════════════════════»
                                                                               »
«     ┌───────────────┐┌───────────────┐┌───────────────┐                 »
«q_0: ┤ Ry(theta[27]) ├┤ Rz(theta[28]) ├┤ Rx(theta[29]) ├─────────────────»
«     └───────────────┘├───────────────┤├───────────────┤┌───────────────┐»
«q_1: ────────■────────┤ Ry(theta[30]) ├┤ Rz(theta[31]) ├┤ Rx(theta[32]) ├»
«             │        └───────────────┘├───────────────┤├───────────────┤»
«q_2: ────────┼────────────────■────────┤ Ry(theta[33]) ├┤ Rz(theta[34]) ├»
«             │                │        └───────────────┘├───────────────┤»
«q_3: ────────┼────────────────┼────────────────■────────┤ Ry(theta[36]) ├»
«             │                │                │        └───────────────┘»
«q_4: ────────■────────────────┼────────────────┼────────────────■────────»
«                              │                │                │        »
«q_5: ─────────────────────────■────────────────┼────────────────┼────────»
«                                               │                │        »
«q_6: ──────────────────────────────────────────■────────────────┼────────»
«                                                                │        »
«q_7: ───────────────────────────────────────────────────────────■────────»
«                                                                         »
«q_8: ────────────────────────────────────────────────────────────────────»
«                                                                         »
«c: 1/════════════════════════════════════════════════════════════════════»
«                                                                         »
«                                                                            »
«q_0: ────────■────────────────────────────────────────────────────────────■─»
«             │                                                            │ »
«q_1: ────────■────────────────■───────────────────────────────────────────┼─»
«     ┌───────────────┐        │                                           │ »
«q_2: ┤ Rx(theta[35]) ├────────■───────────────────────────────────────────┼─»
«     ├───────────────┤┌───────────────┐                                   │ »
«q_3: ┤ Rz(theta[37]) ├┤ Rx(theta[38]) ├─────────────────────────■─────────■─»
«     ├───────────────┤├───────────────┤┌───────────────┐        │          

Now we create a function that will take in parameters and determine the loss, so we can simply pass this to scipy.optimize.minimize and have the optimization done for us. We choose to use mean square error as our loss function for simplicity here.

In [ ]:
from qiskit.primitives import StatevectorSampler

def mse(predictions, labels):
  diff = np.array(predictions) - labels
  return (diff @ diff) / len(diff)

shots = 1000

sampler = StatevectorSampler(seed=42)


def get_expectations(qc : QuantumCircuit, features, param_vec_phi: ParameterVector, num_shots=1000):
  expectations = []

  for feat in features:
    qc_bound = set_encoding(feat, qc, param_vec_phi)

    result = sampler.run([qc_bound], shots=num_shots).result()

    counts = result[0].data.c.get_counts()

    p1 = counts.get("1", 0) / num_shots
    expectations.append(1-2*p1)

  return expectations

# print(get_expectations(circuits, x_train))


def get_loss(parameters, param_vec_theta: ParameterVector, param_vec_phi: ParameterVector, qc: QuantumCircuit, features, labels, num_shots=100):
  # Set theta
  qc_bound = set_params(parameters, qc, param_vec_theta)

  # Get expectations
  expectations = get_expectations(qc_bound, features, param_vec_phi, num_shots)

  # Return loss
  return mse(expectations, labels)

print(get_loss(np.zeros(len(theta_main)), theta_main, phi_main, main, x_train, y_train))




1.6150857142857142


Now all that's left is to run the optimizer!

In [ ]:
from scipy.optimize import minimize
import time


rng = np.random.default_rng(42)
ansatz = rng.random(len(theta_main))


start_time = time.perf_counter()
angles  = minimize(get_loss, ansatz, args=(theta_main, phi_main, main, x_train, y_train), method="COBYLA")

end_time = time.perf_counter()
print(f"Training time: {end_time-start_time} seconds")

Training time: 300.06436539100014 seconds


In [ ]:
print(angles)

print(f"weights: {angles.x}")

#print(f"loss: {get_loss(angles.x, theta_main, circuits, x_train, y_train)}")

 message: Return from COBYLA because the objective function has been evaluated MAXFUN times.
 success: False
  status: 3
     fun: 0.5482274285714285
       x: [ 2.970e+00 -1.315e-01 ...  1.040e+00 -3.202e-02]
    nfev: 1000
   maxcv: 0.0
weights: [ 2.97025346 -0.13154856  1.19739786  1.83307332  1.12591982  3.01922968
 -0.18284779  1.30870754 -0.18866974  0.01929079  1.5902247   2.00350198
  1.53904763  0.32309387  0.02205368  0.41535846  1.45097828 -0.26415391
  2.16642136  0.01194811  1.65377774  0.43151702  1.53897023  1.6583364
 -0.08621458  0.97438595 -0.01405804 -0.6896129   1.03615449  0.30882786
  2.07293724  1.10710898  2.06873669  0.17159873  2.18224043  1.46694879
  0.04517321  0.28837049 -0.32751801  0.6546533   1.46876656  1.53288251
  1.52834279  0.75481583  1.71516348  2.54000122  1.73288523  0.11488586
  0.7004122   0.01729006  1.57591687  1.15928688  1.61993778  0.51980303
  0.64285384  0.94169532  0.66491705  0.2132249   1.65035571 -0.16630039
  1.02615559  0.6042879

We can also check if it classifies correctly.

In [ ]:
optimal_circuit = set_params(angles.x, main, theta_main)

output = get_expectations(optimal_circuit, x_train, phi_main)
for y_hat, y in zip(output, y_train):
  if abs(y_hat) < 0: # Deprecated
    print(f"Low confidence. Actual: {y}")
  elif y_hat * y >= 0:
    print("Correct!")
  else:
    print(f"Wrong :(. Guess: {y_hat}. Actual: {y}")

Correct!
Correct!
Wrong :(. Guess: -0.038000000000000034. Actual: 1.0
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!


Now out of sample:

In [ ]:
validation_expectations = get_expectations(optimal_circuit, x_validate, phi_main)
for y_hat, y in zip(validation_expectations, y_validate):
  if abs(y_hat) < 0: # Deprecated
    print(f"Low confidence. Actual: {y}")
  elif y_hat * y >= 0:
    print("Correct!")
  else:
    print(f"Wrong :(. Guess: {y_hat}. Actual: {y}")

Correct!
Wrong :(. Guess: -0.08800000000000008. Actual: 1.0
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!


If we switch to the commented x_train and y_train, we can take a smaller circuit and analyze how the number of shots affects training time, number of loss evaluations, and minimum loss achieved.

In [ ]:
import matplotlib.pyplot as plt

main_test, phi_test, theta_test = buildCircuit(1)

shots_test = [i**2 for i in range(10, 100, 10)]

loss_test = []

training_times = []

num_evals = []

for num_shots in shots_test:

  # Get training time
  start_time = time.perf_counter()

  angles = minimize(get_loss, rng.random(len(theta_test)), args=(theta_test, phi_test, main_test, x_train, y_train, num_shots), method="COBYLA")

  end_time = time.perf_counter()

  training_times.append(end_time - start_time)

  loss_test.append(angles.fun)

  # How many times getLoss was called
  num_evals.append(angles.nfev)


KeyboardInterrupt: 

In [ ]:
fig, axs = plt.subplots(1, 3)


axs[0].scatter(shots_test, training_times)
# axs[0].xlabel("Number of Shots")
# axs[0].ylabel("Training Time (Seconds)")
# axs[0].ylim(bottom = 0)
# axs[0].xlim(left=0)

axs[1].scatter(shots_test, num_evals)
axs[2].scatter(shots_test, loss_test)
plt.show()